# Shielding design with HVL and TVL

Size a slab shield the fast way: half-value layers, tenth-value layers,
and narrow-beam transmission through one or more materials.

**Goals**

1. Compute linear attenuation coefficients from the bundled NIST XCOM data.
2. Use HVL / TVL arithmetic for first-pass shield sizing.
3. Stack layers with `multilayer_transmit`.
4. Recognize when narrow-beam arithmetic is not enough.

**Sources**

* NIST XCOM (photon cross sections): https://physics.nist.gov/PhysRefData/Xcom/html/xcom1.html
* NIST X-Ray Mass Attenuation Coefficients (material compositions):
  https://physics.nist.gov/PhysRefData/XrayMassCoef/tab4.html
* Beer-Lambert law: `I = I0 * exp(-mu * x)`


## Beer-Lambert, HVL, and TVL

In the narrow-beam (unidirectional, no scatter) approximation,

$$I = I_0 e^{-\mu x}, \qquad \mathrm{HVL} = \frac{\ln 2}{\mu}, \qquad
\mathrm{TVL} = \frac{\ln 10}{\mu}.$$

`mu(E)` comes from NIST XCOM total attenuation with coherent scattering
(https://physics.nist.gov/PhysRefData/Xcom/html/xcom1.html), bundled for
seven materials. Caveat: this is *narrow beam*. Broad-beam designs need a
buildup factor, which pydecay adds in v0.7 - treat the numbers below as a
lower bound on the transmitted dose until then.


In [ ]:
import math

from pydecay import hvl_slab, mu_from_material, transmit_slab, tvl_slab

E = 1.25  # MeV, mean Co-60 gamma energy

mu_concrete = mu_from_material("concrete", E)
hvl = hvl_slab("concrete", E)
tvl = tvl_slab("concrete", E)
print(f"concrete at {E} MeV: mu = {mu_concrete:.4f} 1/m, HVL = {hvl:.4f} m, TVL = {tvl:.4f} m")

through_hvl = transmit_slab(1.0, "concrete", hvl, E)
print(f"transmission through exactly one HVL: {through_hvl:.6f}  (expect 0.5)")
assert math.isclose(through_hvl, 0.5, rel_tol=1e-12)


## Stacking layers and sizing to a target

`multilayer_transmit(I0, [(material, thickness), ...], E)` multiplies the
individual transmissions - order does not matter for pure attenuation.
To hit a target transmission $T$ with $n$ tenth-value layers:

$$n = rac{\log_{10}(1/T)}{1}, \qquad x = n 	imes \mathrm{TVL}.$$


In [ ]:
from pydecay import multilayer_transmit

stack = multilayer_transmit(1.0, [("lead", 0.01), ("concrete", 0.20)], E)
print(f"1 cm lead + 20 cm concrete at {E} MeV: transmission = {stack:.6e}")

target_T = 1e-3
n_tvl = math.log10(1 / target_T)
x_design = n_tvl * tvl
t_check = transmit_slab(1.0, "concrete", x_design, E)
print(f"design for 1e-3: {n_tvl:.0f} TVL = {x_design:.4f} m -> {t_check:.6e}")
assert math.isclose(t_check, target_T, rel_tol=1e-12)


## Exercise: design a concrete tunnel wall

A Co-60 beam (treat as 1.25 MeV narrow beam) must be reduced to
**1/1000** of its intensity. How much concrete is required? Compute the
thickness from TVL arithmetic and verify with `transmit_slab`.

The solution is in the next cell, tagged `answer`.


In [ ]:
# ANSWER (cell tagged "answer" - hide while attempting the exercise)
t_needed = 3 * tvl_slab("concrete", 1.25)  # 3 TVLs = 10^-3
t_ratio = transmit_slab(1.0, "concrete", t_needed, 1.25)
print(f"concrete thickness: {t_needed:.4f} m (3 TVL)")
print(f"transmission: {t_ratio:.6e}")
assert math.isclose(t_ratio, 1e-3, rel_tol=1e-12)
print("exercise checks passed")


## Golden check

Pinned in `tests/golden/golden_values.json`: transmission through exactly
one HVL of concrete at 1.25 MeV equals 1/2, and the double-thickness slab
transmission equals the square of the single slab (both identities of the
Beer-Lambert law with the bundled NIST XCOM `mu`). `tests/test_notebooks.py`
compares every value printed below against that file.


In [ ]:
import json
import math

half = transmit_slab(1.0, "concrete", hvl_slab("concrete", 1.25), 1.25)
single = transmit_slab(1.0, "concrete", 0.05, 1.25)
double = transmit_slab(1.0, "concrete", 0.10, 1.25)
double_ratio = double / single**2

assert math.isclose(half, 0.5, rel_tol=1e-12)
assert math.isclose(double_ratio, 1.0, rel_tol=1e-12)

payload = {
    "hvl_half_intensity_identity": half,
    "slab_double_thickness_identity": double_ratio,
}
print("GOLDEN_CHECK " + json.dumps(payload))
